<a href="https://colab.research.google.com/github/oliwialosko/ML_Assignment1/blob/main/assignments/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/oliwialosko/ML_Assignment1/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## Finding 1: The Essence of Freshness

The paper reports a over 3x health boost and a 57x impression boost for older content (365+ days) that received a recent refresh.  

**Methodology Question:** How was the refreshed cohort selected? If editorial teams naturally choose to refresh only their highest-potential, historically successful pages (leaving truly dead pages untouched), the 57x impression boost might be influenced by human selection rather than the effect of the update itself.

## Finding 2: The Content Performance Curve

Content naturally hits peak performance at 61-90 days post-publication before eventually dropping off.  

**Methodology Question:** Does this 61-90 day peak reflect the search discovery lifecycle (Google's algorithms), or is it influenced by the promotional cadences of the 57 tracked brands? For example, the traffic spike might be promotion-driven.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

I already run the grouped split (by client_hash_id) in week 5, but to prove its importancy I will now show the differency (before/after) and how it can be biased when the split is too good to be true.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, getpass
import duckdb
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from xgboost import XGBClassifier

In [ ]:
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token: ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

TARGET_MONTH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

In [ ]:
query = f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_past,
        SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_clicks ELSE 0 END) AS clicks_past,
        AVG(CASE WHEN report_date <= '2026-03-15' THEN NULLIF(gsc_avg_position, 0) END) AS avg_pos_past,
        SUM(CASE WHEN report_date > '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_future
    FROM read_parquet('{TARGET_MONTH}')
    GROUP BY 1, 2
    HAVING imp_past >= 100
"""
df = con.sql(query).df()
df['ctr_past'] = np.where(df['imp_past'] > 0, (df['clicks_past'] / df['imp_past']) * 100, 0)
df['is_declining_label'] = (df['imp_future'] < 0.8 * df['imp_past']).astype(int)
df = df.dropna(subset=['avg_pos_past']).copy()

features = ['imp_past', 'ctr_past', 'avg_pos_past']
target = 'is_declining_label'

def precision_at_k(labels, scores, k=50):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
#standard train test split - no group shuffle
X_train_rnd, X_val_rnd, y_train_rnd, y_val_rnd = train_test_split(
    df[features], df[target], test_size=0.25, random_state=42
)
xgb_rnd = XGBClassifier(n_estimators=100, random_state=42, max_depth=4, learning_rate=0.1)
xgb_rnd.fit(X_train_rnd, y_train_rnd)
p50_rnd = precision_at_k(y_val_rnd, xgb_rnd.predict_proba(X_val_rnd)[:, 1], k=50)

In [ ]:
#group shuffle split
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, val_idx = next(gss.split(df, groups=df['client_hash_id']))

X_train_grp = df.iloc[train_idx][features]
y_train_grp = df.iloc[train_idx][target]
X_val_grp = df.iloc[val_idx][features]
y_val_grp = df.iloc[val_idx][target]

xgb_grp = XGBClassifier(n_estimators=100, random_state=42, max_depth=4, learning_rate=0.1)
xgb_grp.fit(X_train_grp, y_train_grp)
p50_grp = precision_at_k(y_val_grp, xgb_grp.predict_proba(X_val_grp)[:, 1], k=50)


In [ ]:
print("COMPARISON: \n")
print(f"Naive Random Split Precision@50:   {p50_rnd:.3f}")
print(f"Honest Grouped Split Precision@50: {p50_grp:.3f}")

COMPARISON: 

Naive Random Split Precision@50:   0.700
Honest Grouped Split Precision@50: 0.200


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Executing the leakage on the final model:

- Future/Overlapping Windows: Cleared. All features (_past) are strictly bounded by the dates <= 2026-03-15. The target label (_future) is strictly calculated from the future window > 2026-03-15. There is zero temporal overlap.

- Decision-derived features: Cleared. The model uses only raw search metrics (impressions, clicks, positions).

- Label-derived features: Cleared. The true future answer (imp_future) was strictly excluded.

The cell below verifies that feature importance is distributed reasonably

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
importances = pd.DataFrame({
    'Feature': features,
    'Importance': xgb_grp.feature_importances_
}).sort_values('Importance', ascending=False)

print("If a feature is > 0.90, it might be a leak ")
print(importances.to_markdown(index=False))

If a feature is > 0.90, it might be a leak 
| Feature      |   Importance |
|:-------------|-------------:|
| ctr_past     |     0.620499 |
| avg_pos_past |     0.219604 |
| imp_past     |     0.159897 |


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original boldest claim:**

My XGBoost model identifies which pages will lose traffic. Because past CTR is the dominant feature, pages with low click-through rates will inevitably drop in performance, meaning one must rewrite their snippets immediately to save the traffic.

**Rewritten Safe Claim:**

In the holdout validation, it is shown that pages with a low historical click-through rate show a directional tendency toward future traffic decline. The current XGBoost model provides a measured risk score based on these historical signals. It should be used as a decision-support tool to help the editorial team prioritize the refresh queue, rather than a guaranteed prediction of traffic drop.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.